# ST-OMR Meter V4-2 Full-Train Candidate + Development Screen

Trains one deterministic numerator candidate on the exact 27 accepted V4-0 TRAIN crops, repeats training for determinism, and only then decodes the 9 pre-existing Teacher Gold adaptation-validation positives for a development-only screen. D10, TEST, runtime and production remain closed.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
from hashlib import sha256
import json, shutil, subprocess, sys

REPO_URL = 'https://github.com/khfy7wpr5p-maker/st-omr-training.git'
REPO_REF = 'fix/meter-v4-2-full-train-dev-screen'
WORK_ROOT = Path('/content/st-omr-meter-v4-2')
REPO_DIR = WORK_ROOT / 'repo'
PARENT_V4_0_ROOT = Path('/content/drive/MyDrive/TEST/METER_V1/02_ADAPTATION_RUNS/meter-v4-0-numerator-representation-audit-8641fc45ae0e')
PARENT_V4_1_ROOT = Path('/content/drive/MyDrive/TEST/METER_V1/02_ADAPTATION_RUNS/meter-v4-1-learned-numerator-specialist-b780052b0482')
PILOT_ROOT = Path('/content/drive/MyDrive/TEST/METER_V1/00_AUDIT/teacher_gold_pilot_v1')
RUNS_ROOT = Path('/content/drive/MyDrive/TEST/METER_V1/02_ADAPTATION_RUNS')

if WORK_ROOT.exists():
    shutil.rmtree(WORK_ROOT)
WORK_ROOT.mkdir(parents=True)
subprocess.run(['git','clone','--branch',REPO_REF,'--single-branch','--filter=blob:none',REPO_URL,str(REPO_DIR)], check=True)
git_sha = subprocess.run(['git','-C',str(REPO_DIR),'rev-parse','HEAD'], check=True, capture_output=True, text=True).stdout.strip()
if len(git_sha) != 40 or any(ch not in '0123456789abcdef' for ch in git_sha):
    raise RuntimeError('Git SHA is not canonical lowercase SHA-1')
repository_binding = sha256(('git-commit-sha1:' + git_sha).encode('ascii')).hexdigest()
OUTPUT_ROOT = RUNS_ROOT / f'meter-v4-2-full-train-dev-screen-{repository_binding[:12]}'

required = [
    PARENT_V4_0_ROOT/'result.json', PARENT_V4_0_ROOT/'COMPLETE', PARENT_V4_0_ROOT/'crops',
    PARENT_V4_1_ROOT/'result.json', PARENT_V4_1_ROOT/'COMPLETE',
    PILOT_ROOT/'pilot-data.json',
    PILOT_ROOT/'ST_OMR_METER_TEACHER_GOLD_PILOT_choices.json',
    PILOT_ROOT/'meter-training-permission-evidence-v1.json',
    PILOT_ROOT/'meter-privacy-review-evidence-v1.json',
]
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError('Missing required artifact(s): ' + repr(missing))

print(json.dumps({
    'branch': REPO_REF,
    'git_commit_sha': git_sha,
    'repository_sha256_binding': repository_binding,
    'parent_train_crops': 27,
    'development_validation_images': 9,
    'development_validation_used_for_training': False,
    'd10_opened': False,
    'test_opened': False,
    'output_root': str(OUTPUT_ROOT),
}, indent=2))


## Install exact pinned CPU training runtime


In [ ]:
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    '--extra-index-url', 'https://download.pytorch.org/whl/cpu',
    '-r', str(REPO_DIR/'requirements-training.txt')
], check=True)
runtime = subprocess.run([sys.executable, '-c', 'import torch; print(torch.__version__)'], check=True, capture_output=True, text=True).stdout.strip()
print('SUBPROCESS TORCH:', runtime)
if runtime != '2.13.0+cpu':
    raise RuntimeError(f'Pinned Torch mismatch: {runtime}')


## Run with live stage output


In [ ]:
result_path = OUTPUT_ROOT/'result.json'
complete_path = OUTPUT_ROOT/'COMPLETE'
if result_path.is_file() and complete_path.is_file():
    print('V4-2 already COMPLETE; reusing immutable result.')
else:
    if OUTPUT_ROOT.exists():
        raise RuntimeError(f'Incomplete V4-2 output exists; do not overwrite: {OUTPUT_ROOT}')
    part = OUTPUT_ROOT.with_name('.' + OUTPUT_ROOT.name + '.part')
    if part.exists():
        raise RuntimeError(f'Incomplete V4-2 temporary output exists; do not overwrite: {part}')
    command = [
        sys.executable, '-u', str(REPO_DIR/'tools/meter_v4_2_full_train_dev_screen_runner.py'),
        '--repository-root', str(REPO_DIR),
        '--parent-v4-0-root', str(PARENT_V4_0_ROOT),
        '--parent-v4-1-root', str(PARENT_V4_1_ROOT),
        '--pilot', str(PILOT_ROOT/'pilot-data.json'),
        '--choices', str(PILOT_ROOT/'ST_OMR_METER_TEACHER_GOLD_PILOT_choices.json'),
        '--permission', str(PILOT_ROOT/'meter-training-permission-evidence-v1.json'),
        '--privacy', str(PILOT_ROOT/'meter-privacy-review-evidence-v1.json'),
        '--output-root', str(OUTPUT_ROOT),
    ]
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    assert process.stdout is not None
    for line in process.stdout:
        print(line.rstrip(), flush=True)
    code = process.wait()
    if code != 0:
        raise RuntimeError(f'V4-2 runner failed with exit code {code}')


## Bounded result


In [ ]:
if not result_path.is_file() or not complete_path.is_file():
    raise RuntimeError('V4-2 result/COMPLETE missing')
result = json.loads(result_path.read_text(encoding='ascii'))
print('==============================================')
print('V4-2 DEVELOPMENT SUMMARY')
print('==============================================')
print(json.dumps({k:v for k,v in result['development_validation'].items() if k != 'predictions'}, indent=2, ensure_ascii=False))
print('\n==============================================')
print('V4-2 DECISION')
print('==============================================')
print(json.dumps(result['decision'], indent=2, ensure_ascii=False))
print('\n==============================================')
print('V4-2 SAFETY')
print('==============================================')
print(json.dumps({
    **result['safety'],
    'deterministic_repeat_pass': result['full_train']['deterministic_repeat_pass'],
    'candidate_checkpoint_sha256': result['candidate_checkpoint']['sha256'],
    'candidate_model_state_sha256': result['candidate_checkpoint']['model_state_sha256'],
}, indent=2))
